<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Predict credit-card default

**[Predict credit-card default](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/)** · Machine Learning: From Problem to Reliable Model · Module 9, case study 3 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** build a credit-risk model end to end on real data: check the data, frame the problem, set baselines, compare models in an experiment log, choose a threshold by cost, check calibration and fairness slices, and test once.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 1 minute. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Modules 1 to 8 of this course, most of all [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/), [Slice analysis](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/) and [When a simple model is better](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/). |
| **Data** | UCI *Default of Credit Card Clients* (Yeh, 2009), CC BY 4.0, https://doi.org/10.24432/C55S3H. Real, anonymous clients of one bank in Taiwan, 2005. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L03-predict-credit-card-default/predict-credit-card-default-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It downloads `credit_default_taiwan.csv.gz` (about 1 MB) from the Nextia labs repository into a folder `credit-default`, and checks its checksum. It needs an internet connection and no account. If a copy of the file is next to this notebook, the cell uses that copy.

You should see `Ready: credit_default_taiwan.csv.gz, checksum OK.`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `credit-default` (in Colab: the **Files** panel on the left), then run the cell again.

The data is real and anonymous: 30,000 credit-card clients of a bank in Taiwan in 2005. Data: "Default of Credit Card Clients" by I-Cheng Yeh, UCI Machine Learning Repository, https://doi.org/10.24432/C55S3H, CC BY 4.0. Nextia Learning converted it from XLS to a gzip CSV, removed the extra first header row and renamed the target column to `default`.

In [ ]:
# Setup: download the data file into credit-default/ and check it. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/credit_default_taiwan.csv.gz"
FILE, SHA256 = "credit_default_taiwan.csv.gz", "a773cfc32f6759261cebfdfdc4f744bce1b247a583fe7c06363a12702c98f727"
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("credit-default").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

target = WORK / FILE
if not (target.exists() and sha256(target) == SHA256):
    if (HERE / FILE).exists():
        shutil.copy(HERE / FILE, target)
    else:
        try:
            urllib.request.urlretrieve(URL, target)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(target), URL]).returncode:
                raise SystemExit(f"Could not download {URL}. Check your internet connection.")
if sha256(target) != SHA256:
    raise SystemExit(f"{FILE} has the wrong checksum. Delete the folder credit-default and run again.")
os.chdir(WORK)  # work in this folder: the experiment log and the model are saved here
print(f"Ready: {FILE}, checksum OK.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, numpy as `np` and matplotlib as `plt`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Mei-Ling Chen leads the card-risk team of a bank. Every month, before the next statement, her team chooses clients for an **early-help call**: an agent offers a payment plan and pauses any limit increase. A call to a client who would have paid anyway costs about **1 unit** (agent time, and an annoyed good client). A client who defaults without a call costs about **5 units** (the part of the debt that the bank does not recover). Her analyst, Arjun, builds the model. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/) tells the full story; this notebook runs every step.

## 2. The data

Load the file and look at its size and columns.

In [ ]:
raw = pd.read_csv("credit_default_taiwan.csv.gz")
print(raw.shape)
print(raw.columns.tolist())

> **Check.** You should see `(30000, 25)`: an `ID`, the credit limit, four personal facts, six status columns `PAY_0, PAY_2, …, PAY_6`, six bill amounts, six payment amounts and the target `default`.

**What this tells us.** One row is one client. The six months run from September (`PAY_0`, `BILL_AMT1`, `PAY_AMT1`) back to April 2005. The target is whether the client defaulted on the October payment.

Now the first checks: missing values, repeated rows and the default rate.

In [ ]:
print("missing values:", int(raw.isna().sum().sum()))
print("repeated IDs:", int(raw["ID"].duplicated().sum()))
print("same row, other ID:", int(raw.drop(columns="ID").duplicated().sum()))
print("defaults:", int(raw["default"].sum()), "of", len(raw), f"({raw['default'].mean():.1%})")

> **Check.** You should see `missing values: 0`, `repeated IDs: 0`, `same row, other ID: 35` and `defaults: 6636 of 30000 (22.1%)`.

**What this tells us.** No gaps. The 35 repeated rows are mostly clients with no activity at all; they have different IDs, so we keep them. About one client in five defaults: rare enough that accuracy would mislead, common enough for a model to learn from.

Next, the codes. The source documents `EDUCATION` 1–4, `MARRIAGE` 1–3, and the status −1 (paid duly) and 1–9 (months of delay).

> **Predict.** Will every code in the file be one of the documented ones?

In [ ]:
for column in ["EDUCATION", "MARRIAGE", "PAY_0", "PAY_2"]:
    print(column, raw[column].value_counts().sort_index().to_dict())
print("negative September bills:", int((raw["BILL_AMT1"] < 0).sum()))

> **Check.** You should see `EDUCATION` with 0, 5 and 6; `MARRIAGE` with 0; the status with −2 and 0 (14,737 clients have 0 in September); and 590 negative September bills.

**What this tells us.** The file has codes that nobody documented. −2 and 0 are the most common status values, so we cannot drop them: we keep each status code as a **category**, not as a number. Education 0, 5 and 6 go into "others" (4). Look also at code 1: it appears in 3,688 September values but in only 28 August values. September is coded differently from the other months.

> **Your turn.** How many clients have an undocumented education code (0, 5 or 6)? Put the number in `undocumented_education`, then run the check cell.

In [ ]:
undocumented_education = int(raw["EDUCATION"].isin([0, 5, 6]).sum())
print(undocumented_education)

In [ ]:
expect_hash('undocumented_education', undocumented_education, 'da70dfa4d9f95ac9')

## 3. Frame the problem

`prepare()` renames `PAY_0` to `PAY_1`, so that month 1 is September in every column, and groups the undocumented education codes. Then we name the columns. **Sex, marital status and age are never inputs**: they stay in the data only to check the model for each group later. Run the cell.

In [ ]:
def prepare(raw):
    """Rename PAY_0 to PAY_1 (September), and put the undocumented
    education codes 0, 5 and 6 into 4 (others)."""
    clients = raw.rename(columns={"PAY_0": "PAY_1"})
    clients["EDUCATION"] = clients["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
    return clients


clients = prepare(raw)
STATUS = [f"PAY_{m}" for m in range(1, 7)]  # repayment status, 1 = September ... 6 = April
AMOUNTS = ["LIMIT_BAL"] + [f"BILL_AMT{m}" for m in range(1, 7)] + [f"PAY_AMT{m}" for m in range(1, 7)]
CATEGORIES = ["EDUCATION"] + STATUS
FEATURES = CATEGORIES + AMOUNTS
SENSITIVE = ["SEX", "MARRIAGE", "AGE"]  # never inputs: only for slice checks
TARGET = "default"
print(len(FEATURES), "features | kept out of the inputs:", SENSITIVE)

> **Check.** You should see `20 features | kept out of the inputs: ['SEX', 'MARRIAGE', 'AGE']`.

**What this tells us.** All 20 inputs are known on the statement date, before October: they describe the last six months. Nothing from October is an input, so the model has no leak.

## 4. Explore what the decision needs

The team's question is "who will not pay?". First: how does the default rate change with the September status?

In [ ]:
by_status = clients.groupby(clients["PAY_1"].clip(upper=3))[TARGET].agg(clients="size", default_rate="mean")
print(by_status.round(3))

> **Check.** You should see six rows, from −2 to 3 (3 means 3 months or more). The rate is 0.128 to 0.168 for −2, −1 and 0, then 0.339 for 1 and 0.691 for 2.

**What this tells us.** A delay in September is the strongest single sign: two months late means about 7 defaults in 10. But −2, −1 and 0 are not in order of risk (0 is the lowest), which confirms that the codes are categories. The chart shows the same rates.

In [ ]:
ax = by_status["default_rate"].plot.bar(color="#3b6fb6", rot=0)
ax.set_xlabel("Repayment status in September (3 = 3 months late or more)")
ax.set_ylabel("Default rate in October")
plt.show()

Next, the amounts and the credit limit. Run the cell.

In [ ]:
print(clients[["LIMIT_BAL", "BILL_AMT1", "PAY_AMT1"]].agg(["median", "mean", "max"]).round(0))
limit_band = pd.cut(clients["LIMIT_BAL"], [0, 50_000, 140_000, 240_000, 1_000_000],
                    labels=["up to 50k", "50k-140k", "140k-240k", "over 240k"])
print(clients.groupby(limit_band, observed=True)[TARGET].agg(clients="size", default_rate="mean").round(3))

> **Check.** The median September payment is 2,100 NT dollars, the mean 5,664 and the largest 873,552. The default rate falls from 0.318 (limit up to 50,000) to 0.140 (over 240,000).

**What this tells us.** A few very large amounts would dominate a linear model, so the pipeline takes a log first. The credit limit matters too. Remember it: the limit is the bank's own earlier decision, and it comes back in the fairness checks.

## 5. Split and baselines

There is one snapshot and no date per client, so a time split is not possible. A random split, stratified by the target, matches the use: score this month's clients like the ones the model learned from. Train 60%, validation 20%, test 20%. The test set stays sealed until section 10.

In [ ]:
from sklearn.model_selection import train_test_split

rest, test = train_test_split(clients, test_size=0.2, stratify=clients[TARGET], random_state=0)
train, valid = train_test_split(rest, test_size=0.25, stratify=rest[TARGET], random_state=0)
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} {len(part):6d} clients, {part[TARGET].sum():5d} defaults ({part[TARGET].mean():.1%})")

> **Check.** You should see 18,000, 6,000 and 6,000 clients, each with 22.1% defaults.

Now the cost. `report()` counts the four outcomes of a yes/no flag, and the total cost: 5 for each missed default, 1 for each false alarm. Then three baselines on the validation set: call nobody, call everybody, and the rule the team uses today, "call every client who was late in September".

> **Predict.** Which baseline is the cheapest?

In [ ]:
MISS_COST, ALARM_COST = 5, 1


def report(y, flags):
    """Flags against the real answers: counts, precision, recall and the cost."""
    y, flags = np.asarray(y, dtype=bool), np.asarray(flags, dtype=bool)
    caught, false_alarms, missed = int((flags & y).sum()), int((flags & ~y).sum()), int((~flags & y).sum())
    cost = MISS_COST * missed + ALARM_COST * false_alarms
    return {"flagged": round(flags.mean(), 3), "caught": caught, "false alarms": false_alarms, "missed": missed,
            "precision": round(caught / max(caught + false_alarms, 1), 3), "recall": round(caught / y.sum(), 3),
            "cost": cost, "cost per 1,000": round(1000 * cost / len(y))}


y_valid = valid[TARGET]
baselines = pd.DataFrame([report(y_valid, np.zeros(len(valid))), report(y_valid, np.ones(len(valid))),
                          report(y_valid, valid["PAY_1"] >= 1)],
                         index=["call nobody", "call everybody", "rule: late in September"])
print(baselines.to_string())

> **Check.** Call nobody costs 6,635 (1,327 missed × 5); call everybody 4,673; the rule **3,934**, with recall 0.511 and precision 0.496.

**What this tells us.** The rule is already much better than nobody or everybody. The model must beat **3,934** on the same 6,000 clients, or it is not worth building.

## 6. Models and the experiment log

The pipeline has two branches: the status codes and education go through one-hot encoding (codes rarer than 50 clients in train share one column), and the amounts through a signed log and scaling. `signed_log` keeps negative bills negative.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def signed_log(values):
    """log(1 + |x|) with the sign of x: it shrinks large amounts and keeps negative bills negative."""
    return np.sign(values) * np.log1p(np.abs(values))


def build_pipeline(categories=CATEGORIES, amounts=AMOUNTS, log=True, C=1.0):
    """Status codes one-hot (rare codes grouped), amounts log-scaled and standardised, then logistic regression."""
    amount_steps = [("log", FunctionTransformer(signed_log, feature_names_out="one-to-one"))] if log else []
    amount_steps.append(("scale", StandardScaler()))
    prepare_step = ColumnTransformer([
        ("status", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), categories),
        ("amounts", Pipeline(amount_steps), amounts),
    ])
    return Pipeline([("prepare", prepare_step), ("model", LogisticRegression(max_iter=2000, C=C))])

Every experiment is scored the same way: average precision (AP) on 5 stratified folds of train. `log_experiment()` writes one row of `experiment_log.csv`, with the columns of [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/). Run the cell. It prints nothing.

In [ ]:
from datetime import date
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
LOG = Path("experiment_log.csv")
COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]
pd.DataFrame(columns=COLUMNS).to_csv(LOG, index=False)  # start an empty log


def cv_ap(pipeline, columns, data=None):
    """AP of a pipeline on each of the 5 folds of train."""
    data = train if data is None else data
    return cross_val_score(pipeline, data[columns], data[TARGET], cv=cv, scoring="average_precision")


def rule_ap(score_of):
    """AP of a fixed rule (it learns nothing) on the same 5 folds."""
    return np.array([average_precision_score(train[TARGET].iloc[rows], score_of(train.iloc[rows]))
                     for _, rows in cv.split(train, train[TARGET])])


def log_experiment(id, change, folds, decision, reason, valid_ap=None):
    row = {"id": id, "date": date.today().isoformat(), "change": change, "cv_ap": round(folds.mean(), 3),
           "cv_std": round(folds.std(), 3), "valid_ap": valid_ap, "decision": decision, "reason": reason}
    pd.DataFrame([row]).to_csv(LOG, mode="a", header=False, index=False)
    print(f"{id}  CV AP {folds.mean():.3f} ± {folds.std():.3f}  {decision}: {change}")

Three first rows: the floor (every client gets the same score), today's rule as a 0/1 score, and logistic regression with one feature, the September status.

> **Predict.** The floor is 0.221. How far above it will the rule be? The one-feature model?

In [ ]:
log_experiment("E01", "Baseline: one score for every client (the train rate)",
               rule_ap(lambda part: np.full(len(part), train[TARGET].mean())), "baseline", "The floor.")
log_experiment("E02", "Baseline: the rule 'late in September' as a 0/1 score",
               rule_ap(lambda part: (part["PAY_1"] >= 1).astype(int)), "baseline", "What the team does today.")
log_experiment("E03", "Logistic regression, PAY_1 only (one-hot)", cv_ap(build_pipeline(["PAY_1"], []), ["PAY_1"]),
               "reject", "Above the rule, but it ignores five months of history.")

> **Check.** You should see CV AP 0.221, 0.367 and 0.453.

**What this tells us.** The September status alone, as categories, already beats the rule: the model gives a different risk to each code, while the rule only says yes or no.

Now all 20 features, in three versions: the status codes as numbers (a mistake on purpose), the amounts without the log, and the full pipeline.

In [ ]:
log_experiment("E04", "All 20 features, status codes as numbers",
               cv_ap(build_pipeline(["EDUCATION"], AMOUNTS + STATUS), FEATURES),
               "reject", "The codes -2, -1, 0 are not amounts; as numbers they lose information.")
log_experiment("E05", "All 20 features, amounts without the log", cv_ap(build_pipeline(log=False), FEATURES),
               "reject", "Lower than E06 on 4 of 5 folds.")
log_experiment("E06", "All 20 features, status one-hot, signed log of amounts", cv_ap(build_pipeline(), FEATURES),
               "keep", "Best so far; train AP close to CV, so it does not overfit.")

> **Check.** You should see 0.495, 0.536 and **0.542**.

**What this tells us.** The status as categories is worth 0.047 AP (E04 against E06): the cost of the undocumented codes, if you treat them as numbers. The log adds 0.006, higher on 4 of 5 folds: small, but cheap.

> **Your turn.** Arjun suggests two new features: `utilisation` = `BILL_AMT1 / LIMIT_BAL`, and `paid_share` = `PAY_AMT1 / BILL_AMT2` (with `BILL_AMT2` clipped at 1 from below, so that you never divide by 0). Add them to a copy of train, and score `build_pipeline(amounts=AMOUNTS + ["utilisation", "paid_share"])` with `cv_ap(..., columns, data=extra)`. Put the mean of the 5 folds, rounded to 3 decimals, in `e07_ap`. Then log it as E07 with your decision.

In [ ]:
extra = train.assign(utilisation=train["BILL_AMT1"] / train["LIMIT_BAL"],
                     paid_share=train["PAY_AMT1"] / train["BILL_AMT2"].clip(lower=1))
e07_folds = cv_ap(build_pipeline(amounts=AMOUNTS + ["utilisation", "paid_share"]),
                  FEATURES + ["utilisation", "paid_share"], data=extra)
e07_ap = round(e07_folds.mean(), 3)
log_experiment("E07", "E06 + utilisation and paid share", e07_folds, "reject",
               "No gain (0.541 against 0.542): two more columns to explain for nothing.")

In [ ]:
expect_hash('e07_ap', e07_ap, '96874b11492319b3')

**What this tells us.** A feature that sounds useful is not always useful: the model already sees the bill, the payment and the limit, so the ratios add nothing it cannot combine itself.

Next, the regularization setting `C`, with a small grid search on the same folds.

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(build_pipeline(), {"model__C": [0.01, 0.1, 1, 10]}, cv=cv, scoring="average_precision")
search.fit(train[FEATURES], train[TARGET])
grid = pd.DataFrame(search.cv_results_)[["param_model__C", "mean_test_score", "std_test_score"]]
print(grid.round(4).to_string(index=False))
log_experiment("E08", "E06 with C = 10, the best of the grid", cv_ap(build_pipeline(C=10), FEATURES), "reject",
               "C = 0.1, 1 and 10 tie (0.541 to 0.542); keep the default C = 1 (E06).")

> **Check.** You should see 0.5349, 0.5413, 0.5416 and 0.5417, then the row E08 with CV AP 0.542 ± 0.007.

**What this tells us.** The setting does not matter here: with 18,000 clients and 55 prepared columns, the model has little room to overfit.

Now two experiments that test a policy, not only a score. First, add sex, marital status and age as inputs. Second, a gradient-boosting model as a challenger.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

log_experiment("E09", "E06 + SEX, MARRIAGE and AGE as inputs",
               cv_ap(build_pipeline(CATEGORIES + ["SEX", "MARRIAGE"], AMOUNTS + ["AGE"]), FEATURES + SENSITIVE),
               "reject", "Policy: personal facts are not inputs; the gain (0.001) is smaller than the noise.")
e10_folds = cv_ap(HistGradientBoostingClassifier(random_state=0), FEATURES)
print("E10 - E06 by fold:", (e10_folds - cv_ap(build_pipeline(), FEATURES)).round(3))

> **Check.** E09 shows 0.543. The fold differences of boosting against E06 are `[ 0.005  0.014  0.013  0.013 -0.004]`.

**What this tells us.** The personal facts add almost nothing: the history already carries the signal. Boosting is better on 4 of 5 folds, by about 0.008 AP. Before we decide, we check both on the validation set, once.

In [ ]:
from sklearn.metrics import brier_score_loss, roc_auc_score

pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
booster = HistGradientBoostingClassifier(random_state=0).fit(train[FEATURES], train[TARGET])
booster_scores = booster.predict_proba(valid[FEATURES])[:, 1]
for name, s in [("logistic regression (E06)", scores), ("boosting (E10)", booster_scores)]:
    print(f"{name:26s} ROC AUC {roc_auc_score(y_valid, s):.3f}  AP {average_precision_score(y_valid, s):.3f}")

> **Check.** Logistic regression: ROC AUC 0.775, AP 0.555. Boosting: ROC AUC 0.783, AP 0.559.

**What this tells us.** Boosting ranks a little better (+0.008 ROC AUC). Section 8 turns that into money. We log E10 as a challenger that we do not use, and fill the valid AP of E06. Then print the whole log.

In [ ]:
log_experiment("E10", "Gradient boosting (HistGradientBoosting, defaults), 20 features", e10_folds, "reject",
               "Small gain (+0.008 AP); it cannot give each client exact reasons. Kept as a challenger.",
               valid_ap=round(average_precision_score(y_valid, booster_scores), 3))
log = pd.read_csv(LOG)
log.loc[log["id"] == "E06", "valid_ap"] = round(average_precision_score(y_valid, scores), 3)
log.to_csv(LOG, index=False)
print(log[["id", "change", "cv_ap", "cv_std", "valid_ap", "decision"]].to_string(index=False))

> **Check.** You should see ten rows, E01 to E10. E06 is the only `keep`, with valid AP 0.555.

## 7. Metric and operating point: a threshold from the costs

A score is not a decision. A threshold turns it into "call" or "do not call", and here the costs choose it, not a capacity. Try every threshold from 0.01 to 0.99 on the validation set, and keep the cheapest.

> **Predict.** Is the cheapest threshold above or below 0.5?

In [ ]:
THRESHOLDS = np.round(np.arange(0.01, 1.0, 0.01), 2)
costs = pd.Series([report(y_valid, scores >= t)["cost"] for t in THRESHOLDS], index=THRESHOLDS)
threshold = float(costs.idxmin())
print("cheapest threshold:", threshold, "| cost:", int(costs.min()))
table = pd.DataFrame([report(y_valid, scores >= t) for t in [0.5, 0.3, 0.2, threshold, 0.1]],
                     index=[0.5, 0.3, 0.2, threshold, 0.1])
print(table.to_string())

> **Check.** You should see `cheapest threshold: 0.17 | cost: 3355`. At 0.17 the model flags 39.9% of clients, catches 946 of 1,327 defaults (recall 0.713) and costs 3,355. At 0.5 it costs 4,458.

**What this tells us.** The model at 0.17 costs 14.7% less than today's rule (3,934) and 24.7% less than the same model at the default 0.5. The threshold is low because a miss costs five times a false alarm. With calibrated scores, theory says: call when the chance of default is above 1 ÷ (1 + 5) = 0.167. The data agrees.

In [ ]:
ax = costs.loc[0.05:0.6].plot(color="#3b6fb6")
ax.axvline(threshold, linestyle="--", color="gray")
ax.set_xlabel("Threshold (call when the score is at least this)")
ax.set_ylabel("Total cost on valid (miss 5, false alarm 1)")
plt.show()

The curve is flat from 0.14 to 0.17 (3,375 to 3,355), so the exact value matters little. It rises fast above 0.2.

> **Your turn: change one thing.** Mei-Ling says that a missed default may cost only **3** units. Find the cheapest threshold with `MISS_COST = 3` (use the same `THRESHOLDS` and `report()`), and put it in `threshold_3`. Then set `MISS_COST` back to 5. Predict first: up or down?

In [ ]:
MISS_COST = 3
costs_3 = pd.Series([report(y_valid, scores >= t)["cost"] for t in THRESHOLDS], index=THRESHOLDS)
threshold_3 = float(costs_3.idxmin())
MISS_COST = 5
print(threshold_3, int(costs_3.min()))

In [ ]:
expect_hash('threshold_3', threshold_3, 'a30a043314fa8929')

**What this tells us.** A cheaper miss moves the threshold up (theory: 1 ÷ (1 + 3) = 0.25), so the team makes fewer calls. The threshold belongs to the business's costs: when the costs change, the threshold changes, and the model does not.

### Calibration

The cost rule "call above 0.167" works only if a score of 0.17 means about a 17% chance. Check it on the validation set.

In [ ]:
bins = [0, 0.1, 0.2, 0.3, 0.5, 0.7, 1]
calibration = pd.DataFrame({"score": scores, "default": y_valid.to_numpy()}).groupby(
    pd.cut(scores, bins), observed=True).agg(clients=("default", "size"), mean_score=("score", "mean"),
                                             default_rate=("default", "mean"))
print(calibration.round(3))
print("Brier:", round(brier_score_loss(y_valid, scores), 4), "| mean score", round(scores.mean(), 3),
      "| default rate", round(y_valid.mean(), 3))

> **Check.** In each bin, the mean score and the default rate are close: 0.138 against 0.143, 0.239 against 0.229, 0.605 against 0.628. The Brier score is 0.1346; the mean score 0.223 and the rate 0.221.

**What this tells us.** The scores can be read as chances on this data, so the cost rule is sound. Unlike Larkfield's model after the policy change, nothing changed between train and validation here: both come from the same month.

### Reasons for one client

A client who gets the call can ask "why me?". For logistic regression, each client's score splits into contributions. `contributions()` sums them into six reason groups, and we compare one client with the average train client.

In [ ]:
REASON_OF = {"PAY_1": "late status in September", "LIMIT_BAL": "credit limit", "EDUCATION": "education",
             **{f"PAY_{m}": "late status, April to August" for m in range(2, 7)},
             **{f"BILL_AMT{m}": "bill amounts" for m in range(1, 7)},
             **{f"PAY_AMT{m}": "payments made" for m in range(1, 7)}}


def contributions(pipeline, part):
    """Coefficient times prepared value, summed into the six reason groups, one row per client."""
    prepare_step, model = pipeline.named_steps["prepare"], pipeline.named_steps["model"]
    values = np.asarray(prepare_step.transform(part[FEATURES])) * model.coef_[0]
    names = [n.split("__", 1)[1] for n in prepare_step.get_feature_names_out()]
    columns = [next(c for c in FEATURES if n == c or n.startswith(c + "_")) for n in names]
    table = pd.DataFrame(values, columns=[REASON_OF[c] for c in columns], index=part.index)
    return table.T.groupby(level=0).sum().T

Now client 3008 of the validation set. Run the cell.

In [ ]:
average = contributions(pipeline, train).mean()
client = valid.assign(score=scores)[valid["ID"] == 3008]
print(client[["ID", "LIMIT_BAL"] + STATUS + ["PAY_AMT1", "PAY_AMT2", "score"]].to_string(index=False))
reasons = (contributions(pipeline, client).iloc[0] - average).sort_values(ascending=False)
print(reasons.round(2).to_string())

> **Check.** Client 3008 has a limit of 10,000, was 1 month late in September and 2 months late in every month from April to August, paid nothing in September, and has a score of 0.728. The largest reasons are `late status, April to August` 0.87, `payments made` 0.72 and `credit limit` 0.46.

**What this tells us.** The agent can say: "You were two months late in each of the last five months, and the last payment was zero." This client paid in October, so the call was a false alarm; the reasons explain the **score**, not what the client will do.

## 8. Errors

Where does the model fail at 0.17? Look at the missed defaults and the false alarms.

In [ ]:
errors = valid.assign(score=scores, flag=scores >= threshold)
missed = errors[~errors["flag"] & (errors[TARGET] == 1)]
false_alarms = errors[errors["flag"] & (errors[TARGET] == 0)]
print("missed defaults:", len(missed), "| no delay in any of the 6 months:", int((missed[STATUS] <= 0).all(axis=1).sum()),
      "| median score", round(missed["score"].median(), 3))
print("false alarms:", len(false_alarms), "| late in September:", int((false_alarms["PAY_1"] >= 1).sum()),
      "| limit up to 50,000:", int((false_alarms["LIMIT_BAL"] <= 50_000).sum()))

> **Check.** You should see 381 missed defaults, 350 of them with no delay in any month, median score 0.126; and 1,450 false alarms, 639 late in September and 568 with a limit up to 50,000.

**What this tells us.** Almost every missed default (350 of 381) looked like a good payer for six months: no column in this data can warn about a sudden default. The false alarms are clients who were late but paid, and many clients with low limits. The low limits lead to the next question.

## 9. Fairness slices

`slice_table()` gives, for each group, the clients, the default rate, the share flagged, recall, the **false-alarm rate** (the share of good payers who get the call) and ROC AUC.

In [ ]:
AGE_BANDS, AGE_LABELS = [20, 25, 30, 40, 50, 80], ["21-25", "26-30", "31-40", "41-50", "51+"]


def slice_table(part, part_scores, column, threshold=threshold):
    """Clients, default rate, flag rate, recall, false-alarm rate and ROC AUC per group."""
    frame = part.assign(score=part_scores, flag=part_scores >= threshold)
    rows = {}
    for group, g in frame.groupby(column, observed=True):
        good, bad = g[g[TARGET] == 0], g[g[TARGET] == 1]
        rows[group] = {"clients": len(g), "default rate": round(g[TARGET].mean(), 3),
                       "flagged": round(g["flag"].mean(), 3), "recall": round(bad["flag"].mean(), 3),
                       "false-alarm rate": round(good["flag"].mean(), 3),
                       "ROC AUC": round(roc_auc_score(g[TARGET], g["score"]), 3) if g[TARGET].nunique() == 2 else None}
    return pd.DataFrame(rows).T


valid_bands = valid.assign(age_band=pd.cut(valid["AGE"], AGE_BANDS, labels=AGE_LABELS))
for column in ["SEX", "MARRIAGE", "age_band"]:
    print(slice_table(valid_bands, scores, column).to_string(), end="\n\n")

> **Check.** By sex (1 male, 2 female) the false-alarm rates are 0.323 and 0.302. Marital status 0 has only 8 clients. Age 21–25: 822 clients, default rate 0.260, flagged **0.523**, false-alarm rate **0.428**; the other bands have 0.279 to 0.330.

**What this tells us.** Age is not an input, yet young clients are flagged far more often. Part of it is real (26.0% of them default, against 21.2% for 31–40), but good payers aged 21–25 get the call 1.5 times as often as good payers aged 31–40. The model must be using another column that goes with age. A guess: the credit limit.

In [ ]:
print(valid_bands.groupby("age_band", observed=True)["LIMIT_BAL"].median())
low = (valid_bands["LIMIT_BAL"] <= 50_000).to_numpy()
print("\nlimit up to 50,000:\n", slice_table(valid_bands[low], scores[low], "age_band")[["clients", "default rate", "false-alarm rate"]])
print("\nlimit over 50,000:\n", slice_table(valid_bands[~low], scores[~low], "age_band")[["clients", "default rate", "false-alarm rate"]])

> **Check.** The median limit is 50,000 for 21–25 and 100,000 to 180,000 for the others. Within the same limit band, the false-alarm rates are close: 0.478 to 0.573 up to 50,000, and 0.223 to 0.259 over 50,000.

**What this tells us.** This is a **proxy**: the limit stands in for age. 59% of clients aged 21–25 have a limit of 50,000 or less, and the model treats a low limit as a risk. The limit was set by the bank itself, so the model repeats the bank's earlier decision about young clients. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/) discusses what the team can do. One option is to drop `LIMIT_BAL`.

> **Your turn.** Fit `build_pipeline(amounts=no_limit)` on train without `LIMIT_BAL`, score valid, and find the false-alarm rate of the band 21–25 at the same threshold with `slice_table()`. Put it in `alarm_young_no_limit` (a float with 3 decimals), and the total cost in `cost_no_limit`.

In [ ]:
no_limit = [c for c in AMOUNTS if c != "LIMIT_BAL"]
p_no_limit = build_pipeline(amounts=no_limit).fit(train[CATEGORIES + no_limit], train[TARGET])
s_no_limit = p_no_limit.predict_proba(valid[CATEGORIES + no_limit])[:, 1]
alarm_young_no_limit = float(slice_table(valid_bands, s_no_limit, "age_band").loc["21-25", "false-alarm rate"])
cost_no_limit = report(y_valid, s_no_limit >= threshold)["cost"]
print(alarm_young_no_limit, cost_no_limit)

In [ ]:
expect_hash('(alarm_young_no_limit, cost_no_limit)', (alarm_young_no_limit, cost_no_limit), 'f1ac8b3af076ed53')

**What this tells us.** Without the limit, the false-alarm rate of young clients falls from 0.428 to 0.373, and the cost rises from 3,355 to 3,425 (2%). The gap shrinks but does not close: the payment history also differs by age. Whether 2% is a fair price is a decision for the business and its compliance team, not for the analyst alone.

## 10. The final test, once

Every choice is made: the model E06, the threshold 0.17, and no change for fairness yet (it is a recommendation). Open the test set once, and compare with the rule on the same clients.

> **Warning.** Run the next cell once. Do not change the model or the threshold after you read it.

In [ ]:
y_test = test[TARGET]
test_scores = pipeline.predict_proba(test[FEATURES])[:, 1]
print("ROC AUC", round(roc_auc_score(y_test, test_scores), 3), "| AP", round(average_precision_score(y_test, test_scores), 3),
      "| Brier", round(brier_score_loss(y_test, test_scores), 4))
final = pd.DataFrame([report(y_test, test_scores >= threshold), report(y_test, test["PAY_1"] >= 1),
                      report(y_test, np.zeros(len(test)))],
                     index=[f"model, threshold {threshold}", "rule: late in September", "call nobody"])
print(final.to_string())

> **Check.** ROC AUC 0.77, AP 0.549. The model costs **3,346** (558 per 1,000 clients), with recall 0.701 and 38.2% flagged; the rule costs 3,852; nobody 6,635.

**What this tells us.** On new clients the model costs 13.1% less than the rule, close to the 14.7% on validation. That is the number to report. The test slices by age are on the lesson page; they are reported, not used to change anything.

### Save the model for the portfolio

Save the pipeline and its threshold, as in [Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/). Run the cell.

In [ ]:
import json
import joblib
import sklearn

joblib.dump(pipeline, "default_model.joblib")
Path("model_info.json").write_text(json.dumps({
    "features": FEATURES, "threshold": threshold, "costs": {"missed_default": MISS_COST, "false_alarm": ALARM_COST},
    "not_inputs": SENSITIVE, "data_sha256": SHA256, "scikit-learn": sklearn.__version__}, indent=2))
print(sorted(p.name for p in Path(".").iterdir() if not p.name.startswith(".")))

> **Check.** You should see `['credit_default_taiwan.csv.gz', 'default_model.joblib', 'experiment_log.csv', 'model_info.json']`. These files, this notebook and a README make the portfolio piece: the lesson page has the template.

## Next

You have built a credit-risk model end to end: a baseline to beat, an experiment log, a threshold from the costs, a calibration check, reasons for one client, a proxy found with slices, and one honest test. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/) has the recommendations, the limits, the portfolio template and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Rank a call list](https://learning.nextia-ai.com/courses/ml/m09/rank-a-call-list/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and anonymous, from the UCI Machine Learning Repository (CC BY 4.0).